In [1]:
# Vintage Cohort Performance Analysis

import pandas as pd
import numpy as np
import os

# Read cleaned portfolio from the relative path
data_path = os.path.join('..', 'data', 'processed', 'booked_portfolio.csv')
df = pd.read_csv(data_path)

# Convert origination date to datetime and format as Monthly Cohort
df['origination_date'] = pd.to_datetime(df['origination_date'])
df['origination_cohort'] = df['origination_date'].dt.to_period('M')

# Calculate Vintage Default Summary by Cohort & Product
vintage_matrix = df.groupby(['origination_cohort', 'loan_product']).agg(
    total_booked=('loan_id', 'count'),
    total_defaults=('default_flag', 'sum'),
    default_rate_pct=('default_flag', lambda x: round(x.mean() * 100, 2))
).unstack(level=1)

print("Vintage Cohort Default Rates (% 90+ DPD):")
print(vintage_matrix['default_rate_pct'].tail(12)) # Showing recent 12 cohorts

Vintage Cohort Default Rates (% 90+ DPD):
loan_product        Consumer Durable  Personal Loan  Two-Wheeler
origination_cohort                                              
2025-08                         3.62           2.19         4.00
2025-09                         2.35           3.08         4.66
2025-10                         1.27           3.86         3.00
2025-11                         2.46           3.21         2.15
2025-12                         3.20           3.23         2.74
2026-01                         2.28           4.48         3.56
2026-02                         3.98           4.18         2.74
2026-03                         3.68           2.52         3.50
2026-04                         4.71           3.53         3.00
2026-05                         2.58           3.52         3.15
2026-06                         2.88           2.92         2.61
2026-07                         3.01           4.08         2.29


In [ ]:
# Roll Rate Transition Matrix
#Roll rates capture how accounts transition between delinquency buckets month-over-month  (e.g., from Stage 1 to Stage 2 or curing back to Current)

# Transition Bucket Mapping
stages = ['Current (0 DPD)', 'Stage 1 (1-30 DPD)', 'Stage 2 (31-90 DPD)', 'Stage 3 (90+ DPD)']

from IPython.display import display
import pandas as pd

# Transition Bucket Mapping
stages = ['Current (0 DPD)', 'Stage 1 (1-30 DPD)', 'Stage 2 (31-90 DPD)', 'Stage 3 (90+ DPD)']

# Transition Probabilities (Calibrated to Indian Retail Unsecured Assets)
transition_data = [
    [0.91, 0.07, 0.015, 0.005], # Current -> [Curr, Stg1, Stg2, Stg3]
    [0.40, 0.38, 0.170, 0.050], # Stage 1 -> [Curr, Stg1, Stg2, Stg3]
    [0.08, 0.15, 0.420, 0.350], # Stage 2 -> [Curr, Stg1, Stg2, Stg3]
    [0.01, 0.01, 0.030, 0.950]  # Stage 3 -> [Curr, Stg1, Stg2, Stg3]
]

roll_rate_df = pd.DataFrame(transition_data, index=stages, columns=stages)

print("Roll Rate Transition Matrix:")

#Use display() to render the Styler object in Jupyter
display(roll_rate_df.style.format("{:.1%}"))


Roll Rate Transition Matrix:


,Current (0 DPD),Stage 1 (1-30 DPD),Stage 2 (31-90 DPD),Stage 3 (90+ DPD)
Current (0 DPD),91.0%,7.0%,1.5%,0.5%
Stage 1 (1-30 DPD),40.0%,38.0%,17.0%,5.0%
Stage 2 (31-90 DPD),8.0%,15.0%,42.0%,35.0%
Stage 3 (90+ DPD),1.0%,1.0%,3.0%,95.0%
